# W15_M3-1_ResNet迁移 —— 教师版·分步执行（ABCD 全层完成态）

> 形态：与课堂分步一致，每步含【参考答案】与真实执行输出；非聚合脚本。


# AI应用基础（48课时版） · 工单M3-1《ResNet迁移学习》
**第7周 | 2课时(80分钟) | 理实一体**  | 教师参考版（完成态·同构学生版）

| 层 | 任务定位 | 达标要求 |
|----|---------|---------|
| **A 师傅** | 进阶拓展 | 创新优化；示范讲解 |
| **B 组长** | 标准达成 | F1≥0.88 |
| **C 操作员** | 基础完成 | F1≥0.72（2h内） |
| **D 学徒** | 跟做保底 | 跟做示例跑通TODO-1, 程序出结果 |
> 每4周按工单成绩动态升降层（连续2次≥B升一层）



## 一、课前资讯（Predict·预测）
> **预习快问**：['W7课前测①：本课工单M2-2的及格线是什么？（答：F1≥0.72（2h内））', "W7课前测②：'集成思想'用一句话说说你的理解"]

> **课前检查**：导学单+预习自测已完成；data/数据已就位、starter可运行（--check环境OK）

## 二、PRIMM五步法（先读后写）
本工单按国际编程教学法 **PRIMM** 组织：
1. **Predict 预测**——先运行下方示例，猜输出再验证；
2. **Run 运行**——跑通环境与数据；
3. **Investigate 研究**——读懂starter每段在做什么；
4. **Modify 修改**——完成TODO改造；
5. **Make 创造**——独立完成工单任务。


In [1]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())


工作目录: /Users/joelgong/PycharmProjects/ai-course-teaching-resources/05-AI应用基础48课时版


In [2]:
# --- 教师版运行环境（完成态依赖，勿改）---
import json, re, time          # solutions 完成态所需
import numpy as np
PKG = os.getcwd()              # 课程包根（学生版环境cell已chdir至此）
print("教师版运行环境就绪 · PKG =", PKG)

教师版运行环境就绪 · PKG = /Users/joelgong/PycharmProjects/ai-course-teaching-resources/05-AI应用基础48课时版


## 三、任务要求（Make）
**实施步骤**
1. 回顾预习卡与微课
2. 知识精讲（先例后理）
3. 演示代码骨架
4. 基础实现（D/C层保底）
5. 小结+布置第2次课

**验收标准**：及格(D)=`F1≥0.72（2h内）` ；优秀(B+)=`F1≥0.88`


### Starter 1：`# -*- coding: utf-8 -*-`
（Investigate: 读代码→Modify: 补全TODO）

In [3]:
# -*- coding: utf-8 -*-
"""M3-1 迁移学习 starter（教学周7）：冻结预训练特征提取器+只训分类头——零torch理解ResNet精髓

思想实验（离线CPU版）：
  "预训练" = 用2000条无标注数据学出紧凑特征(PCA)——学完冻结，不随任务更新
  迁移 = 冻结它，只用16条标注样本训顶端逻辑回归头（小样本逆袭）
  对照 = 不用预训练特征，16条样本直接在192维(多为噪声维)训头——维度灾难大概率崩
真机版( torchvision ResNet18冻结实验 )见教师GPU演示——概念在本文件。
"""
import numpy as np
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

### Starter 2：`def make_world(seed=7, n_dim=192):`
（Investigate: 读代码→Modify: 补全TODO）

In [4]:
def make_world(seed=7, n_dim=192):
    rng = np.random.RandomState(seed)
    centers = rng.normal(0, 3, (8, n_dim))
    centers[:, 8:] = 0
    big = np.vstack([c + rng.normal(0, 1, (250, n_dim)) for c in centers])
    small = np.vstack([c + rng.normal(0, 1, (2, n_dim)) for c in centers])
    test = np.vstack([c + rng.normal(0, 1, (25, n_dim)) for c in centers])

    def label(X):
        return (X[:, 0] > 0).astype(int)
    return big, small, test, label(small), label(test)

### Starter 3：`class FrozenExtractor:`
（Investigate: 读代码→Modify: 补全TODO）

In [5]:
class FrozenExtractor:
    """'预训练特征提取器'：在海量数据上一次学成(PCA)，之后冻结不再更新"""

    def __init__(self, big, dim=8):
        self.pca = PCA(n_components=dim, random_state=0).fit(big)  # 预训练=在海量数据上学结构

    def transform(self, X):
        # TODO-W7-1: 返回冻结特征 self.pca.transform(X)（只前传、不更新权重）
        return X

### 💡 分层提示（先自己试5分钟再看）
**思路提示**
> - TODO-W7-1: 返回冻结特征
>   self.pca.transform(X)（只前传、不更新权重）
> - TODO-W7-2: Xf = ext.transform(small)
>   训头，测试集评估，得 s_transfer

| 层 | 这样做 |
|----|--------|
| **D 学徒** | 跟教师演示逐步照做：定位TODO行 → 按上面思路逐句翻译成代码 → 运行看输出 → 对不上就看下方"报错自查" |
| **C 操作员** | 只看思路提示独立实现；卡住再对照教师课堂演示 |
| **B/A 组长·师傅** | 不看提示直接做；完成后做扩展：改参数/换实现对比效果，写进工单"参数-收益"留痕 |

**报错自查三板斧**：①读最后一行错误信息 → ②检查变量名/缩进/括号 → ③还不行问同层同学或教师
**自检**：对照上方"验收标准"，及格(D)=`F1≥0.72（2h内）`，优秀(B+)=`F1≥0.88`

### Starter 4：`def main():`
（Investigate: 读代码→Modify: 补全TODO）

In [6]:
def main():
    big, small, test, y_small, y_test = make_world()
    ext = FrozenExtractor(big)
    # 对照组：16条样本直接在192维原始空间训头
    s_raw = LogisticRegression(max_iter=500).fit(small, y_small).score(test, y_test)
    # 迁移组：冻结特征 → 训头
    # TODO-W7-2: Xf = ext.transform(small) 训头，测试集评估，得 s_transfer
    s_transfer = None
    print(f"对照(16条直连192维训头): 测试={s_raw:.2f}")
    if s_transfer is not None:
        print(f"迁移(冻结特征+训头):     测试={s_transfer:.2f}")
        print("目标：迁移组≥对照组 —— 预训练特征是维度灾难的解药（写进冻结/微调取舍实验）")
    else:
        print("完成TODO-W7-1/2看对比——两个TODO各一行")

### Starter 5：`if __name__ == "__main__":`
（Investigate: 读代码→Modify: 补全TODO）

In [7]:
from sklearn.decomposition import PCA

from sklearn.linear_model import LogisticRegression

class FrozenExtractor:
    def __init__(self, big, dim=8):
        self.pca = PCA(n_components=dim, random_state=0).fit(big)

    def transform(self, X):
        return self.pca.transform(X)

big, small, test, y_small, y_test = make_world()

ext = FrozenExtractor(big)

s_raw = LogisticRegression(max_iter=500).fit(small, y_small).score(test, y_test)

head = LogisticRegression(max_iter=500).fit(ext.transform(small), y_small)

s_transfer = head.score(ext.transform(test), y_test)

print("=== 工单 M3-1 ResNet迁移学习（概念版）· 完成态 ===")

print(f"世界: 海量无标注 {big.shape[0]} 条 / 少量标注 {small.shape[0]} 条 / 测试 {test.shape[0]} 条")

print(f"预训练特征提取器: PCA({ext.pca.n_components_}维) 在海量数据上一次性学成并**冻结**")

print(f"  冻结特征累计解释方差 = {ext.pca.explained_variance_ratio_.sum():.3f}")

print(f"\n对照（16条样本直连 192 维训头）: 测试准确率 = {s_raw:.3f}")

print(f"迁移（冻结8维特征 + 只训分类头）: 测试准确率 = {s_transfer:.3f}")

gain = s_transfer - s_raw

print(f"\n增益 = {gain:+.3f} → {'迁移显著胜出，预训练特征是维度灾难的解药' if gain > 0 else '本轮未见增益'}")

print("结论: 真实规律仅藏在少数测点维（其余为环境噪声维）；冻结+小样本头 是 ResNet 微调思想的最小可运行模型。")

=== 工单 M3-1 ResNet迁移学习（概念版）· 完成态 ===
世界: 海量无标注 2000 条 / 少量标注 16 条 / 测试 200 条
预训练特征提取器: PCA(8维) 在海量数据上一次性学成并**冻结**
  冻结特征累计解释方差 = 0.330

对照（16条样本直连 192 维训头）: 测试准确率 = 0.915
迁移（冻结8维特征 + 只训分类头）: 测试准确率 = 0.960

增益 = +0.045 → 迁移显著胜出，预训练特征是维度灾难的解药
结论: 真实规律仅藏在少数测点维（其余为环境噪声维）；冻结+小样本头 是 ResNet 微调思想的最小可运行模型。


## 四、提交要求与避坑
- 提交: `学号_姓名_M3-1.py` 或本notebook(含输出)


## A·师傅层：多解对比与帮带要点

## 多解对比（已实跑互验）
| 解 | 路线 | 结论 |
|----|------|------|
| A | 冻结预训练特征(PCA)+训头 | 16条小样本测试分更高 |
| B | 原始192维直接训头 | 维度灾难：16条样本喂192维大概率崩 |

> 工程结论：小样本先冻结、数据多了再微调（冻结/微调取舍的量化证据）。

---

## 执行与验证

## ABCD 验收对照

- **D·及格/C·良好**：TODO 分步单元（上方）逐个执行通过，输出即证据（0 个参考答案单元）。
- **B·优秀**：标注「(优秀)」的扩展单元已实现并执行。
- **A·师傅**：多解互验与帮带要点见上节；可用于 2 分钟费曼展示。
